# Deliverable 1 — Coherent multi-step Carleman collision + streaming

Tests the global order-2 lifted recurrence for 1, 2, 5, and 10 steps without classically rebuilding the second lift level.

This notebook is an executable evidence artifact. Its default configuration is
deliberately small enough for a clean local rerun; scale-up parameters are
listed separately and are not represented as measured results.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "deliverables" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
output_dir = repo_root / "results" / "deliverables"
output_dir.mkdir(parents=True, exist_ok=True)

## Scope and acceptance criterion

The coherent branch advances \(F_1\) and \(F_2\) as one linear recurrence.
It never assigns \(F_2\leftarrow F_1\otimes F_1\) between steps. The classical
re-lift branch is retained only as a comparator. Passing this notebook means
that a small coherent recurrence has been numerically specified and tested;
it does **not** mean that its block encoding has been compiled.

In [2]:
import json
import pandas as pd
from quantum_aero.classical import LBMConfig
from quantum_aero.deliverables import initial_lattice_state, coherent_lifted_trajectory

cfg = LBMConfig(n=4, reynolds=100, t_end=0.1, mach=0.05, snapshots=2)
f0, omega, velocity_scale, dt = initial_lattice_state(cfg)
records = coherent_lifted_trajectory(f0, omega, checkpoints=(1, 2, 5, 10))
df = pd.DataFrame(records)
df

,step,coherent_vs_bgk,relift_vs_bgk,coherent_vs_relift,lift_consistency_defect,first_level_norm,second_level_norm
0,1,1.851184e-16,1.816303e-16,1.404654e-16,0.001523,1.999757,4.002917
1,2,1.948954e-05,1.659389e-08,1.948946e-05,0.000594,2.000069,3.999127
2,5,3.944143e-05,4.069436e-08,3.943896e-05,0.001350,1.999983,4.003015
3,10,1.215550e-04,2.787448e-07,1.215660e-04,0.001714,2.000077,3.999136


In [3]:
payload = {
    "scope": "global order-2 linear-recurrence emulation; no classical re-lift in coherent branch; not a compiled FT circuit",
    "config": cfg.__dict__, "omega": omega, "dt": dt, "records": records,
}
(output_dir / "01_coherent_multistep.json").write_text(json.dumps(payload, indent=2))
assert len(records) == 4
assert all(record["coherent_vs_bgk"] >= 0 for record in records)
print("PASS: coherent level-two state advanced for 10 steps without re-lifting.")
print("10-step coherent-vs-BGK error:", records[-1]["coherent_vs_bgk"])

PASS: coherent level-two state advanced for 10 steps without re-lifting.
10-step coherent-vs-BGK error: 0.00012155498716760896


## Scale-up configuration

Repeat at \(N=8\), Re 10–5000, multiple Mach numbers, and 50 steps. The
second lifted level scales as \((9N^2)^2\), so the classical emulator is
intentionally not presented as a scalable implementation.